# Intelligent Spam Classification & Threat Triage Platform
## Notebook 01 — Dataset Exploration

This notebook performs the initial exploratory data analysis (EDA) for the **Intelligent Spam Classification & Threat Triage Platform**.

The objective is to understand the dataset structure, target-label distribution, source composition, data quality, potential bias, and security-specific characteristics before building machine-learning or Agentic AI components.

### Dataset
`puyang2025/seven-phishing-email-datasets`

### Analysis Structure
1. Environment Setup
2. Dataset Loading
3. Dataset Overview
4. Univariate Analysis
5. Bivariate Analysis
6. Data Quality Analysis
7. Security-Oriented Sample Inspection
8. Cross-Split Leakage Check
9. Canonical Schema Preview
10. Key Findings and Modeling Implications

> **Security note:** Treat all email content as untrusted input. Do not click URLs, execute attachments, render untrusted HTML, or run scripts contained in samples.

## 1. Environment Setup

Install and import the minimum dependencies required for dataset exploration.

The notebook is designed to run in **Google Colab**.

In [1]:
!pip -q install datasets pyarrow pandas numpy scikit-learn

In [2]:
import hashlib
import re
import numpy as np
import pandas as pd

from datasets import load_dataset

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42
PROJECT_NAME = "Intelligent Spam Classification & Threat Triage Platform"
DATASET_NAME = "puyang2025/seven-phishing-email-datasets"

print(f"Project : {PROJECT_NAME}")
print(f"Dataset : {DATASET_NAME}")
print(f"Seed    : {RANDOM_STATE}")

Project : Intelligent Spam Classification & Threat Triage Platform
Dataset : puyang2025/seven-phishing-email-datasets
Seed    : 42


## 2. Dataset Loading

The dataset is loaded directly through Hugging Face `datasets`.

The supplied train/test split is preserved so the test split can later serve as a locked final evaluation set.

In [3]:
dataset = load_dataset(DATASET_NAME)
print(dataset)

README.md:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  184MB            

train.parquet: downloading bytes:           |  0.00B            

test.parquet: reconstructing file:   0%|          |  0.00B / 22.7MB            

test.parquet: downloading bytes:           |  0.00B            

eval.parquet: reconstructing file:   0%|          |  0.00B / 22.7MB            

eval.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/162413 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/40604 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name'],
        num_rows: 162413
    })
    test: Dataset({
        features: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name'],
        num_rows: 40604
    })
})


## 3. Dataset Overview

Inspect the available splits, row counts, feature schema, and a truncated sample record.

This ensures downstream processing is designed against the actual source schema.

In [4]:
print("Available splits:")
print(dataset.keys())

for split_name, split in dataset.items():
    print("\n" + "=" * 70)
    print(f"Split: {split_name}")
    print(f"Rows: {len(split):,}")

    print("\nFeatures:")
    print(split.features)

    print("\nFirst record:")
    record = split[0].copy()

    if "text" in record and record["text"] is not None:
        preview = str(record["text"]).replace("\n", " ")
        record["text"] = preview[:300] + ("..." if len(preview) > 300 else "")

    print(record)

Available splits:
dict_keys(['train', 'test'])

Split: train
Rows: 162,413

Features:
{'text': Value('string'), 'subject': Value('string'), 'label': Value('int64'), 'sender': Value('string'), 'receiver': Value('string'), 'date': Value('timestamp[ns]'), 'urls': Value('int64'), 'dataset_name': Value('string')}

First record:
{'text': "       meat helpful round An afterclang of Cowley's mouth chords closed, died on tdesire And Richie like debt Goulding drank his noise Power and Leopold Blsuccessful struck cough Libel action, says he, slow for ten thousand pounds.Very umbrella tame kind hat grip of you, says Bloom. Quavering the ma...", 'subject': 'When will it be okay', 'label': 1, 'sender': 'Lissette Patterson <jstepanekov@eve-team.com>', 'receiver': 'Olympia <dmason@plg2.math.uwaterloo.ca>', 'date': Timestamp('2007-05-02 00:31:14'), 'urls': 0, 'dataset_name': 'TREC-07'}

Split: test
Rows: 40,604

Features:
{'text': Value('string'), 'subject': Value('string'), 'label': Value('int64'), 's

In [5]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")

print("\nColumns:")
print(train_df.columns.tolist())

print("\nData types:")
print(train_df.dtypes)

Train shape : (162413, 8)
Test shape  : (40604, 8)

Columns:
['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name']

Data types:
text                    object
subject                 object
label                    int64
sender                  object
receiver                object
date            datetime64[ns]
urls                   float64
dataset_name            object
dtype: object


# 4. Exploratory Data Analysis

The EDA is organized into explicit analytical categories.

### Univariate Analysis
Examines one variable at a time.

### Bivariate Analysis
Examines relationships between two variables, especially relationships with the target label.

### Data Quality Analysis
Examines missing values, duplicates, annotation inconsistencies, and leakage risks.

### Security-Oriented Inspection
Examines sanitized message samples without interacting with potentially malicious content.

## 4.1 Univariate Analysis

This section examines:
- target-label distribution,
- source-dataset distribution,
- URL indicator distribution,
- text-length distribution,
- subject-length distribution.

The purpose is to identify class imbalance, unusual distributions, and preprocessing requirements.

In [6]:
print("TRAIN LABEL DISTRIBUTION")
print("=" * 50)

train_label_counts = train_df["label"].value_counts().sort_index()
train_label_pct = (
    train_df["label"].value_counts(normalize=True).sort_index().mul(100).round(2)
)

train_label_summary = pd.DataFrame({
    "count": train_label_counts,
    "percentage": train_label_pct
})

print(train_label_summary)

print("\nTEST LABEL DISTRIBUTION")
print("=" * 50)

test_label_counts = test_df["label"].value_counts().sort_index()
test_label_pct = (
    test_df["label"].value_counts(normalize=True).sort_index().mul(100).round(2)
)

test_label_summary = pd.DataFrame({
    "count": test_label_counts,
    "percentage": test_label_pct
})

print(test_label_summary)

TRAIN LABEL DISTRIBUTION
       count  percentage
label                   
0      86951       53.54
1      75462       46.46

TEST LABEL DISTRIBUTION
       count  percentage
label                   
0      21738       53.54
1      18866       46.46


In [7]:
print("SOURCE DATASET DISTRIBUTION")
print("=" * 70)

source_distribution = train_df["dataset_name"].value_counts().to_frame("count")
source_distribution["percentage"] = (
    source_distribution["count"] / len(train_df) * 100
).round(2)

print(source_distribution)

SOURCE DATASET DISTRIBUTION
              count  percentage
dataset_name                   
TREC-05       44461       27.38
TREC-07       42964       26.45
CEAS-08       31150       19.18
Enron         23849       14.68
TREC-06       13113        8.07
Assassin       4585        2.82
Ling           2291        1.41


In [8]:
print("URL FEATURE ANALYSIS")
print("=" * 60)

print("\nData type:")
print(train_df["urls"].dtype)

print("\nMissing:")
print(train_df["urls"].isna().sum())

print("\nDescriptive statistics:")
print(train_df["urls"].describe())

print("\nMost common values:")
print(train_df["urls"].value_counts(dropna=False).head(20))

URL FEATURE ANALYSIS

Data type:
float64

Missing:
26140

Descriptive statistics:
count    136273.000000
mean          0.544062
std           0.498057
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           1.000000
Name: urls, dtype: float64

Most common values:
urls
1.0    74141
0.0    62132
NaN    26140
Name: count, dtype: int64


### Canonical Binary Label Mapping

The source corpus uses a binary ground-truth label.

For the initial ML layer:

- `0 → BENIGN`
- `1 → THREAT`

The positive class is intentionally named **THREAT** because the combined corpus includes both spam-like and phishing-like content.

More specific categories such as `PHISHING`, `BEC`, `CREDENTIAL_PHISHING`, and `MALICIOUS_URL` belong to downstream threat enrichment and Agentic AI.

In [9]:
LABEL_MAP = {
    0: "BENIGN",
    1: "THREAT",
}

train_df["canonical_label"] = train_df["label"].map(LABEL_MAP)
test_df["canonical_label"] = test_df["label"].map(LABEL_MAP)

print("Canonical label mapping:")
print(LABEL_MAP)

print("\nTrain distribution:")
print(train_df["canonical_label"].value_counts())

print("\nTest distribution:")
print(test_df["canonical_label"].value_counts())

Canonical label mapping:
{0: 'BENIGN', 1: 'THREAT'}

Train distribution:
canonical_label
BENIGN    86951
THREAT    75462
Name: count, dtype: int64

Test distribution:
canonical_label
BENIGN    21738
THREAT    18866
Name: count, dtype: int64


In [10]:
train_df["text_length"] = train_df["text"].fillna("").astype(str).str.len()
train_df["subject_length"] = train_df["subject"].fillna("").astype(str).str.len()

test_df["text_length"] = test_df["text"].fillna("").astype(str).str.len()
test_df["subject_length"] = test_df["subject"].fillna("").astype(str).str.len()

print("TRAIN TEXT LENGTH SUMMARY")
print(train_df["text_length"].describe())

print("\nTRAIN SUBJECT LENGTH SUMMARY")
print(train_df["subject_length"].describe())

TRAIN TEXT LENGTH SUMMARY
count    1.624130e+05
mean     1.769954e+03
std      1.101470e+04
min      0.000000e+00
25%      3.440000e+02
50%      7.760000e+02
75%      1.752000e+03
max      2.550619e+06
Name: text_length, dtype: float64

TRAIN SUBJECT LENGTH SUMMARY
count    162413.000000
mean         36.125212
std          31.250878
min           0.000000
25%          20.000000
50%          32.000000
75%          47.000000
max        7170.000000
Name: subject_length, dtype: float64


## 4.2 Bivariate Analysis

This section examines relationships between the target classification and other dataset characteristics.

The main goals are to detect:
- source-specific bias,
- differences in threat rates across corpora,
- structural shortcuts such as message length acting as a proxy for the target.

In [11]:
print("LABEL DISTRIBUTION BY SOURCE")
print("=" * 70)

source_label_distribution = pd.crosstab(
    train_df["dataset_name"],
    train_df["label"],
    margins=True
)

print(source_label_distribution)

LABEL DISTRIBUTION BY SOURCE
label             0      1     All
dataset_name                      
Assassin       3234   1351    4585
CEAS-08       13784  17366   31150
Enron         12643  11206   23849
Ling           1926    365    2291
TREC-05       26029  18432   44461
TREC-06        9904   3209   13113
TREC-07       19431  23533   42964
All           86951  75462  162413


In [12]:
source_stats = (
    train_df
    .groupby("dataset_name")
    .agg(
        total=("label", "size"),
        benign=("label", lambda x: (x == 0).sum()),
        threat=("label", lambda x: (x == 1).sum()),
        threat_rate=("label", "mean"),
    )
)

source_stats["threat_rate_pct"] = (
    source_stats["threat_rate"] * 100
).round(2)

source_stats = source_stats.sort_values("total", ascending=False)

print(source_stats)

              total  benign  threat  threat_rate  threat_rate_pct
dataset_name                                                     
TREC-05       44461   26029   18432     0.414566            41.46
TREC-07       42964   19431   23533     0.547738            54.77
CEAS-08       31150   13784   17366     0.557496            55.75
Enron         23849   12643   11206     0.469873            46.99
TREC-06       13113    9904    3209     0.244719            24.47
Assassin       4585    3234    1351     0.294656            29.47
Ling           2291    1926     365     0.159319            15.93


In [13]:
print("TEXT AND SUBJECT LENGTH BY CLASS")
print("=" * 70)

length_summary = (
    train_df
    .groupby("canonical_label")[["text_length", "subject_length"]]
    .describe()
)

print(length_summary)

TEXT AND SUBJECT LENGTH BY CLASS
                text_length                                                 \
                      count         mean           std  min    25%     50%   
canonical_label                                                              
BENIGN              86951.0  2286.115490  14866.675584  2.0  510.0  1043.0   
THREAT              75462.0  1175.208343   2406.550199  0.0  238.0   555.0   

                                   subject_length                             \
                    75%        max          count       mean        std  min   
canonical_label                                                                
BENIGN           2083.0  2550619.0        86951.0  36.925360  27.836875  0.0   
THREAT           1226.0   230877.0        75462.0  35.203241  34.748830  0.0   

                                           
                  25%   50%   75%     max  
canonical_label                            
BENIGN           21.0  33.0  48.0  3647.0  

In [14]:
median_lengths = (
    train_df
    .groupby("canonical_label")
    .agg(
        median_text_length=("text_length", "median"),
        median_subject_length=("subject_length", "median"),
        mean_text_length=("text_length", "mean"),
        mean_subject_length=("subject_length", "mean"),
    )
    .round(2)
)

print(median_lengths)

                 median_text_length  median_subject_length  mean_text_length  \
canonical_label                                                                
BENIGN                       1043.0                   33.0           2286.12   
THREAT                        555.0                   31.0           1175.21   

                 mean_subject_length  
canonical_label                       
BENIGN                         36.93  
THREAT                         35.20  


### Modeling Interpretation — Potential Shortcut Learning

If BENIGN and THREAT messages have substantially different lengths, a model may learn structural shortcuts rather than security semantics.

Later evaluation should therefore include:
- global metrics,
- per-source metrics,
- feature inspection,
- error analysis,
- comparison between simple ML and Agentic AI reasoning.

## 4.3 Data Quality Analysis

This section checks:
- missing values,
- exact duplicates,
- duplicate bodies,
- duplicate subject/body pairs,
- conflicting labels,
- cross-split overlap.

Cleaning decisions should be explicit and reproducible rather than automatic.

In [15]:
print("MISSING VALUE ANALYSIS — TRAIN")
print("=" * 70)

missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": missing_pct
})

print(missing_summary.sort_values("missing_count", ascending=False))

MISSING VALUE ANALYSIS — TRAIN
                 missing_count  missing_percentage
date                     28898               17.79
sender                   26140               16.09
receiver                 26140               16.09
urls                     26140               16.09
label                        0                0.00
text                         0                0.00
subject                      0                0.00
dataset_name                 0                0.00
canonical_label              0                0.00
text_length                  0                0.00
subject_length               0                0.00


In [16]:
print("DUPLICATE ANALYSIS")
print("=" * 70)

exact_duplicates = train_df.duplicated().sum()
text_duplicates = train_df.duplicated(subset=["text"]).sum()
subject_text_duplicates = train_df.duplicated(subset=["subject", "text"]).sum()

print(f"Exact duplicate rows       : {exact_duplicates:,}")
print(f"Duplicate email bodies     : {text_duplicates:,}")
print(f"Duplicate subject + body   : {subject_text_duplicates:,}")

DUPLICATE ANALYSIS
Exact duplicate rows       : 0
Duplicate email bodies     : 9
Duplicate subject + body   : 1


In [17]:
label_conflicts = (
    train_df
    .groupby("text", dropna=False)["label"]
    .nunique()
)

conflicting_texts = (label_conflicts > 1).sum()

print(f"Email bodies with conflicting labels: {conflicting_texts:,}")

Email bodies with conflicting labels: 0


## 4.4 Security-Oriented Sample Inspection

A small number of BENIGN and THREAT messages are inspected using truncated and defanged previews.

### Safety Principles
- Do not click URLs in samples.
- Do not execute attachments.
- Do not render untrusted HTML.
- Do not run embedded scripts.
- Treat all external email content as potentially hostile.

In [18]:
def sanitize_preview(value, max_chars=300):
    """Return a truncated, display-safe preview of untrusted email text."""

    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""

    text = str(value)
    text = text.replace("\r", " ").replace("\n", " ")

    text = re.sub(
        r"(?i)https?://",
        lambda m: "hxxps://" if m.group(0).lower().startswith("https") else "hxxp://",
        text,
    )

    text = re.sub(r"\s+", " ", text).strip()

    if len(text) > max_chars:
        text = text[:max_chars] + "..."

    return text

In [19]:
sample_columns = [
    "subject",
    "text",
    "sender",
    "receiver",
    "urls",
    "dataset_name",
    "label",
    "canonical_label",
]

samples = (
    train_df
    .groupby("canonical_label", group_keys=False)
    .sample(n=3, random_state=RANDOM_STATE)[sample_columns]
    .copy()
)

samples["subject_preview"] = samples["subject"].apply(
    lambda x: sanitize_preview(x, 150)
)

samples["text_preview"] = samples["text"].apply(
    lambda x: sanitize_preview(x, 300)
)

for _, row in samples.iterrows():
    print("=" * 80)
    print(f"Class   : {row['canonical_label']}")
    print(f"Source  : {row['dataset_name']}")
    print(f"Subject : {row['subject_preview']}")
    print(f"URLs    : {row['urls']}")
    print(f"Preview : {row['text_preview']}")

Class   : BENIGN
Source  : TREC-06
Subject : [DMDX] Re: ANALYZE
URLs    : 0.0
Preview : At 03:37 PM 9/2/2004 +0100, you wrote: >I have successfully run the Unload utility to merge two azk data files >into one. However, I have not been able to run the Analyze program. > >With my merged azk data as the input file, Analyze produces an error >message: not an RTF file, first chars are not "...
Class   : BENIGN
Source  : TREC-07
Subject : Re: Request for suggestions of DFSG-free documentation licenses
URLs    : 1.0
Preview : Shriramana Sharma wrote: >Thanks for all your feedback, but the GPL also has some clauses that are >not applicable to documentation as pointed out at: > >hxxp://www.gnu.org/licenses/gpl-faq.html#WhyNotGPLForManuals Debian does not agree with the FSF opinion on this. The FSF's opinion is basically an...
Class   : BENIGN
Source  : CEAS-08
Subject : [opensuse] [OT] How much power does a PC really consume?
URLs    : 0.0
Preview : This is really off-topic, but as the initial 

## 5. Cross-Split Leakage Check

The supplied train/test split is preserved, but exact overlap must still be checked.

Cross-split duplication can produce misleadingly high evaluation metrics if the same or nearly identical message appears in both sets.

In [20]:
train_text_set = set(train_df["text"].fillna("").astype(str))
test_text_set = set(test_df["text"].fillna("").astype(str))

text_overlap = train_text_set.intersection(test_text_set)

print("CROSS-SPLIT TEXT OVERLAP")
print("=" * 70)
print(f"Unique train texts : {len(train_text_set):,}")
print(f"Unique test texts  : {len(test_text_set):,}")
print(f"Exact text overlap : {len(text_overlap):,}")

CROSS-SPLIT TEXT OVERLAP
Unique train texts : 162,404
Unique test texts  : 40,604
Exact text overlap : 0


In [21]:
train_pairs = set(
    zip(
        train_df["subject"].fillna("").astype(str),
        train_df["text"].fillna("").astype(str),
    )
)

test_pairs = set(
    zip(
        test_df["subject"].fillna("").astype(str),
        test_df["text"].fillna("").astype(str),
    )
)

pair_overlap = train_pairs.intersection(test_pairs)

print("CROSS-SPLIT SUBJECT + BODY OVERLAP")
print("=" * 70)
print(f"Exact subject/body overlap : {len(pair_overlap):,}")

CROSS-SPLIT SUBJECT + BODY OVERLAP
Exact subject/body overlap : 0


## 6. Canonical Schema — Design Preview

The next phase will normalize external dataset fields into a stable internal schema.

| Canonical Field | Source / Purpose |
|---|---|
| `message_id` | Deterministic SHA-256 identifier |
| `subject` | Source `subject` |
| `body` | Source `text` |
| `sender` | Optional source metadata |
| `receiver` | Optional source metadata |
| `timestamp` | Source `date` |
| `has_url` | Normalized source `urls` indicator |
| `source_dataset` | Source `dataset_name` |
| `original_label` | Source integer label |
| `canonical_label` | `BENIGN` or `THREAT` |
| `label_id` | ML-ready integer label |
| `combined_text` | Subject + body |

The reusable implementation will live in:

`src/threat_triage/data_loader.py`

In [22]:
def create_message_id(subject, body, source_dataset):
    payload = "||".join([
        "" if subject is None else str(subject),
        "" if body is None else str(body),
        "" if source_dataset is None else str(source_dataset),
    ])

    return hashlib.sha256(
        payload.encode("utf-8", errors="ignore")
    ).hexdigest()


example_row = train_df.iloc[0]

example_message_id = create_message_id(
    example_row["subject"],
    example_row["text"],
    example_row["dataset_name"],
)

print("Example deterministic message ID:")
print(example_message_id)

Example deterministic message ID:
b87531024f8d5c689ae89b5ec58fc3843f2490f74ceb7b1137bbce1b045298b6


# 7. Key Findings and Modeling Implications

Update this section after executing the notebook and reviewing the actual outputs.

## Class Distribution
Assess whether BENIGN and THREAT are sufficiently balanced.

## Dataset Composition
Compare threat rates across source datasets and note any source-specific bias.

## Missing Metadata
Determine which fields are dependable baseline features and which should remain optional security signals.

## URL Indicator
Confirm whether source `urls` behaves as a binary `has_url` feature and preserve missing values as unknown.

## Potential Message-Length Bias
Compare BENIGN and THREAT structural characteristics and document shortcut-learning risk.

## Ground-Truth Taxonomy
Maintain the binary ML target:
- `BENIGN`
- `THREAT`

Use richer categories only in downstream threat enrichment unless independently labeled.

## Data Split Policy
Preserve the supplied test split as the locked final evaluation set.

Split only the supplied training data into:
- training,
- validation.

## Next Phase
Implement:

`src/threat_triage/data_loader.py`

with reusable functions for:
- dataset loading,
- source-schema validation,
- canonicalization,
- deterministic IDs,
- train/validation/test handling,
- processed-data persistence.

---

## Notebook Completion Checklist

- [ ] Dataset loaded successfully
- [ ] Schema confirmed
- [ ] Train/test shapes verified
- [ ] Label distribution analyzed
- [ ] Source distribution analyzed
- [ ] URL indicator analyzed
- [ ] Canonical labels created
- [ ] Text/subject lengths analyzed
- [ ] Source × label relationship analyzed
- [ ] Missing values analyzed
- [ ] Duplicates analyzed
- [ ] Conflicting labels checked
- [ ] Sanitized samples inspected
- [ ] Cross-split leakage checked
- [ ] Canonical schema reviewed
- [ ] Key findings updated from actual outputs

Once complete, this notebook becomes the project's **EDA baseline**.